# Семинар 02. Словари, множества и хеш-таблицы


## Цели

После семинара вы сможете:

- объяснять устройство хеш-таблицы и причины коллизий;
- оценивать среднюю и худшую сложность операций;
- реализовывать упрощённые set, dict и ordered dict.

## Перед началом

Повторите списки, функции, хешируемые объекты и обозначение O-большое.


## Общая идея

В списке доступ по известному индексу занимает `O(1)`, но поиск произвольного значения требует в худшем случае просмотреть весь список — `O(N)`. Хеш-таблица вычисляет из ключа позицию, где этот ключ следует искать. На этой идее построены `set` и `dict`:

- множество хранит уникальные ключи;
- словарь хранит пары «ключ — значение».

## Путь ключа через хеш-таблицу

```text
ключ ──► hash(ключ) ──► индекс корзины ──► кандидаты ──► сравнение через ==
              │              hash % M              │
              │                                    └── ключ найден / ключа нет
              └── одинаковые ключи обязаны иметь одинаковый хеш
```

В учебной таблице с методом цепочек путь выглядит так. Числа условные:

```text
hash("cat") = 42 ─┐
                     ├── hash % 8 = 2 ──► корзина 2: ["cat", "dog"]
hash("dog") = 18 ─┘                              │
                                                    └── проверить key == candidate

массив корзин:  0: []   1: []   2: [cat, dog]   3: []   ...   7: []
```

Разные ключи могут получить одинаковый хеш или разные хеши могут попасть в один индекс. Поэтому одного хеша недостаточно: после выбора корзины таблица сравнивает ключи через `==`. Хеш ускоряет поиск кандидатов, но не доказывает равенство.

Шаги поиска:

1. Вычислить `hash(key)`.
2. Преобразовать хеш в индекс массива, например `hash(key) % M`.
3. Просмотреть кандидатов в выбранной корзине.
4. Найти равный ключ либо убедиться, что такого ключа нет.


## Коллизии и коэффициент заполнения

Попадание разных ключей в одну позицию называется **коллизией**. Коллизии неизбежны: возможных ключей больше, чем ячеек массива. Таблица обязана их разрешать, иначе данные будут теряться. Два распространённых подхода:

1. **Метод цепочек:** каждая корзина хранит несколько элементов. Именно его нужно реализовать в задании.
2. **Открытая адресация:** элементы лежат в самом массиве, а при занятой позиции алгоритм проверяет другие позиции. `dict` и `set` в CPython используют собственный вариант открытой адресации; это деталь реализации, а не требование языка.

Коэффициент заполнения `α = N / M` связывает число элементов `N` с числом позиций `M`. Если таблица становится слишком плотной, цепочки или последовательности проб растут и операции замедляются. Поэтому реализация периодически создаёт больший массив и перераспределяет элементы. Само перераспределение стоит `O(N)`, но происходит не при каждой вставке.

| Операция | Средняя/амортизированная сложность | Худший случай |
|---|---:|---:|
| поиск по ключу | `O(1)` | `O(N)` |
| вставка | `O(1)` | `O(N)` |
| удаление | `O(1)` | `O(N)` |
| обход всех элементов | `O(N)` | `O(N)` |

Худший случай возникает, когда слишком много ключей сталкиваются или когда вставка вызывает изменение размера. Заявление «операции словаря всегда работают за `O(1)`» технически неверно.

## Какие объекты можно использовать как ключи

Объект хешируем, если его хеш не меняется на протяжении жизни и он поддерживает сравнение. Главный контракт:

```text
a == b  ⇒  hash(a) == hash(b)
```

Обратное неверно: одинаковый хеш не означает равенство. Нарушение контракта ломает поиск — равный ключ может оказаться не в той корзине.

Числа, строки и `frozenset` обычно хешируемы. Списки, словари и обычные множества изменяемы и потому не могут быть ключами. Кортеж хешируем только тогда, когда хешируемы все его элементы.

Хеши строк и байтов в Python по умолчанию рандомизированы между запусками процесса. Значение `hash()` нельзя сохранять в файл или использовать как постоянный идентификатор.

## Поведение встроенного dict

Начиная с Python 3.7 словарь гарантированно сохраняет порядок вставки. Обновление значения существующего ключа не меняет его позицию; после удаления и повторной вставки ключ оказывается в конце. Множество такого контракта порядка не даёт.

Примитивная заготовка таблицы с цепочками:

```python
BUCKET_COUNT = 1009
buckets = [[] for _ in range(BUCKET_COUNT)]


def bucket_index(item) -> int:
    return hash(item) % BUCKET_COUNT


def insert(item) -> None:
    ...


def exists(item) -> bool:
    ...


def delete(item) -> None:
    ...
```

Простое число корзин иногда уменьшает регулярные коллизии при наивном вычислении индекса, но не спасает плохую хеш-функцию. Нормальная таблица обязана корректно обрабатывать любые коллизии.

Полезные источники: [определение hashable](https://docs.python.org/3/glossary.html#term-hashable), [модель данных и `__hash__`](https://docs.python.org/3/reference/datamodel.html#object.__hash__), [порядок словаря](https://docs.python.org/3/library/stdtypes.html#dict).


## Самопроверка

1. Почему после совпадения хеша таблица всё равно сравнивает ключи через `==`?
2. Чем коллизия полного хеша отличается от попадания разных хешей в одну корзину?
3. Почему вставка считается амортизированно `O(1)`, хотя изменение размера стоит `O(N)`?
4. Как рост коэффициента заполнения влияет на цепочки и последовательности проб?
5. Почему список нельзя использовать как ключ словаря, а некоторые кортежи можно?
6. Что произойдёт с порядком словаря после обновления, удаления и повторной вставки ключа?


## Итоги

- Хеш сокращает множество кандидатов, но равенство ключей подтверждает `==`.
- Коллизии неизбежны и должны разрешаться без потери элементов.
- Средняя работа с ключом занимает `O(1)`, худший случай — `O(N)`.
- Изменение размера удерживает коэффициент заполнения под контролем и даёт амортизированную оценку вставки.
- Хешируемый ключ обязан иметь стабильный хеш; равные ключи обязаны иметь одинаковые хеши.
- Встроенный `dict` сохраняет порядок вставки, но это отдельное свойство, а не общее свойство любой хеш-таблицы.


## Задание 1. Собственная реализация set/dict (1 балл)

Допишите упрощённую хеш-таблицу с методом цепочек для разрешения коллизий.

**Сигнатуры**

```python
def insert(item) -> None:
    ...

def exists(item) -> bool:
    ...

def delete(item) -> None:
    ...
```

По желанию обобщите структуру до словаря, хранящего пары ключ—значение.

**Критерии проверки**

- корректно обрабатываются коллизии, повторная вставка и удаление отсутствующего элемента;
- средняя сложность вставки, поиска и удаления — `O(1)`;
- тесты включают элементы с одинаковым индексом корзины.


## Задание 2. Ordered dict (2 балла)

Реализуйте структуру, которая хранит пары ключ—значение и сохраняет порядок первой вставки ключей. Изменение значения не меняет порядок. После удаления и повторной вставки ключ становится последним.

**Интерфейс**

```python
def insert(key, value) -> None:
    ...

def find(key):
    ...

def delete(key) -> None:
    ...

def walk_through():
    ...  # обходит только живые пары в порядке вставки
```

Можно использовать обычный словарь.

**Пример поведения**

```text
insert("a", 1), insert("b", 2), insert("a", 3) -> [("a", 3), ("b", 2)]
delete("a"), insert("a", 4)                 -> [("b", 2), ("a", 4)]
```

**Критерии проверки:** корректный порядок, амортизированное `O(1)` для вставки и удаления, `O(N)` дополнительной памяти.
